# Feature interpretation for CFU prediction model

## Load data

Configure root with local/colab.

In [ ]:
import sys, subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
from pathlib import Path

# Configure root
COLAB = Path("/content").exists()
repo_url = "https://github.com/eddykang06/phenotype-prediction.git"
repo_dir = Path("phenotype-prediction")
if COLAB:
    root = Path("/content/phenotype-prediction")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", repo_url])
else:
    root = Path.cwd().parent
sys.path.insert(0, str(root))

Configure data path for colab/local.

In [ ]:
if COLAB:
  from google.colab import drive
  drive.mount("/content/drive")
  data_dir = Path("/content/drive/MyDrive/phenotype-prediction-data")
  fcnts_path = str(data_dir / "fcnts_timezero")
  cfu_path = str(data_dir /  "cfus")

else:
  fcnts_path = "C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab/fcnts_timezero"
  cfu_path = "C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab/cfus"

Load TPM data and CFU counts.

In [ ]:
from src.tpm_data import get_all_tpm_data

# Get data
data_df = get_all_tpm_data(
    fcnts_path = fcnts_path,
    cfu_path = cfu_path
)

# Find idx where CFU = 0, then list the sample ID
zero_idx = np.where(data_df["CFU"] == 0)[0]
print(data_df.index[zero_idx].tolist())

# Check CFU values for the other 2 replicates
rep_names = ["34CEF4hr-a", "34CEF4hr-b"]
cfu_val_check = data_df.loc[rep_names]["CFU"].tolist()
print(f"CFU values for 34CEF4hr-a and 34CEF4hr-b :{cfu_val_check}")

# Remove sample and convert to log 10 CFU
data_df = data_df[data_df["CFU"] != 0]
data_df["CFU"] = np.log10(data_df["CFU"])

## Feature interpretation for CFU prediction model using forward diagonal

Train model.

In [ ]:
from src.train import train_custom_cfu_model

forward_mask = (data_df["num_drugs"] == 1) | ((data_df["drug1_dose"]) == (data_df["drug2_dose"]))
forward_mask_complement = ~forward_mask

forward_model = train_custom_cfu_model(
    df = data_df,
    train_mask = forward_mask,
    test_mask = forward_mask_complement,
    title = "Results for PLS regression model trained on single-drug and diagonal combination data"
)

Plot coefficient distribution.

In [ ]:
coefs = forward_model.named_steps["model"].coef_.ravel()

plt.hist(coefs)

Rank features by coefficient.

Run GSEA on highest ranked genes.